# 03 — Analyse Qualitative des Erreurs

Ce notebook examine les cas de désaccord entre modèles pour identifier
des **patterns systématiques d'erreurs** selon :

- La **longueur du mail** (court vs long)
- L'**ambiguïté sémantique** (mails à cheval sur deux catégories)
- Le **vocabulaire spécialisé** (jargon sportif ou administratif)

> **Prérequis :** avoir sauvegardé les prédictions via les scripts `run_*.py`
> (les prédictions sont stockées dans les JSON de `results/runs/`)

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path

from src.evaluation.results import RUNS_DIR
from src.data.email_dataset import CLASSES, load_emails
from src.evaluation.metrics import get_classification_report

TEMPLATE = 'plotly_white'

In [2]:
# Chargement du CSV de prédictions généré par scripts/dump_predictions.py
# (entraîne les 4 modèles sur full data / seed 42 et sauvegarde leurs prédictions test)
from pathlib import Path

DATASET = "emails"  # ou "newsgroups"
pred_path = Path("..") / "results" / "aggregated" / f"predictions_{DATASET}.csv"

if not pred_path.exists():
    raise FileNotFoundError(
        f"{pred_path} introuvable.\n"
        f"Lance d'abord : python scripts/dump_predictions.py --dataset {DATASET}"
    )

preds_df = pd.read_csv(pred_path)
pred_cols = [c for c in preds_df.columns if c.startswith("pred_")]
print(f"{len(preds_df)} mails | colonnes prédictions : {pred_cols}")
preds_df.head()

360 mails | colonnes prédictions : ['pred_tfidf_lr', 'pred_camembert', 'pred_setfit_camembert', 'pred_ministral']


,text,true_label,pred_tfidf_lr,pred_camembert,pred_setfit_camembert,pred_ministral
0,"Bonjour,\n\nJe vous contacte car nous serions ...",logistique-matchday,logistique-matchday,logistique-matchday,logistique-matchday,logistique-matchday
1,"Bonjour,\n\nJe dois absolument finaliser les i...",inscription,inscription,inscription,inscription,inscription
2,"Bonjour,\n\nJe vous écris d'urgence car j'aime...",inscription,inscription,inscription,inscription,inscription
3,"Salut,\n\nJ'ai fait les 3 séances cette semain...",indemnites-coachs,indemnites-coachs,indemnites-coachs,indemnites-coachs,indemnites-coachs
4,"Bonjour,\n\nVoilà j'écris pour vous poser une ...",logistique-matchday,logistique-matchday,logistique-matchday,logistique-matchday,parent


---
## 1. Matrices de confusion

Chaque cellule (i, j) = nombre de mails de la classe i classifiés en classe j.
Les erreurs systématiques apparaissent comme des diagonales off-center.

In [3]:
def plot_confusion_matrix(y_true, y_pred, labels, title) -> go.Figure:
    """Heatmap annotée de la matrice de confusion."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    # Normalisation par ligne (recall par classe)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Annotations : count + pourcentage
    annotations = [
        [
            f"{cm[i,j]}<br>({cm_norm[i,j]:.0%})" if cm[i,j] > 0 else "0"
            for j in range(len(labels))
        ]
        for i in range(len(labels))
    ]

    fig = go.Figure(go.Heatmap(
        z=cm_norm,
        x=labels, y=labels,
        colorscale='Blues',
        text=annotations,
        texttemplate='%{text}',
        hovertemplate='Réel: %{y}<br>Prédit: %{x}<br>Recall: %{z:.0%}<extra></extra>',
        zmin=0, zmax=1,
    ))
    fig.update_layout(
        title=title,
        xaxis_title='Prédit',
        yaxis_title='Réel',
        height=500,
        template=TEMPLATE,
    )
    return fig


if 'pred_setfit_camembert' in preds_df.columns and not preds_df['pred_setfit_camembert'].isna().all():
    plot_confusion_matrix(
        preds_df['true_label'], preds_df['pred_setfit_camembert'],
        CLASSES, 'SetFit — Matrice de confusion (emails, full data)'
    ).show()
else:
    print("Prédictions SetFit non disponibles. Lance run_setfit.py d'abord.")

---
## 2. Cas de désaccord entre modèles

Un mail en désaccord est un mail où au moins deux modèles donnent des prédictions
différentes. Ces cas révèlent les zones d'incertitude et les limites de chaque approche.

In [4]:
if len(pred_cols) >= 2:
    preds_df['n_modeles'] = preds_df[pred_cols].notna().sum(axis=1)
    preds_df['n_unique'] = preds_df[pred_cols].apply(
        lambda row: row.dropna().nunique(), axis=1
    )
    preds_df['desaccord'] = preds_df['n_unique'] > 1

    print(f"Désaccords : {preds_df['desaccord'].sum()} / {len(preds_df)} mails "
          f"({preds_df['desaccord'].mean():.0%})")

    # Distribution des désaccords par catégorie réelle
    desacc_rate = (
        preds_df.groupby('true_label')['desaccord'].mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    fig = px.bar(
        desacc_rate, x='desaccord', y='true_label', orientation='h',
        title='Taux de désaccord inter-modèles par catégorie',
        labels={'desaccord': 'Taux de désaccord', 'true_label': ''},
        color='desaccord', color_continuous_scale='Reds',
        template=TEMPLATE, text_auto='.0%'
    )
    fig.update_coloraxes(showscale=False)
    fig.update_xaxes(tickformat='.0%')
    fig.update_layout(height=350)
    fig.show()

Désaccords : 84 / 360 mails (23%)


In [5]:
if len(pred_cols) >= 2 and preds_df['desaccord'].sum() > 0:
    # Affichage de 10 cas de désaccord représentatifs
    sample = preds_df[preds_df['desaccord']].sample(min(10, preds_df['desaccord'].sum()), random_state=42)
    display_cols = ['true_label', 'text'] + pred_cols[:3]
    print(f"\n--- 10 mails en désaccord ---")
    for _, row in sample.iterrows():
        print(f"\n[Réel: {row['true_label']}]")
        for c in pred_cols[:3]:
            if pd.notna(row[c]):
                model_name = c.replace('pred_', '')
                icon = '✓' if row[c] == row['true_label'] else '✗'
                print(f"  {icon} {model_name}: {row[c]}")
        print(f"  Texte: {str(row['text'])[:150]}...")


--- 10 mails en désaccord ---

[Réel: federation]
  ✓ tfidf_lr: federation
  ✓ camembert: federation
  ✓ setfit_camembert: federation
  Texte: Bonjour,

Je vous contacte d'urgence car j'ai reçu aujourd'hui la convocation pour les championnats régionaux d'athlétisme qui auront lieu le 15 avril...

[Réel: logistique-matchday]
  ✓ tfidf_lr: logistique-matchday
  ✓ camembert: logistique-matchday
  ✓ setfit_camembert: logistique-matchday
  Texte: Bonjour,

Voilà j'écris pour vous poser une question à propos du match de samedi... Enfin, c'est surtout pour savoir si il y a besoin de quelque chose...

[Réel: federation]
  ✓ tfidf_lr: federation
  ✓ camembert: federation
  ✓ setfit_camembert: federation
  Texte: Bonjour,

Je dois absolument connaître les dates précises du championnat régional de janvier. Je dois adapter mon planning de travail en urgence. Pouv...

[Réel: inscription]
  ✓ tfidf_lr: inscription
  ✓ camembert: inscription
  ✓ setfit_camembert: inscription
  Texte: Bonjour,

J'esp

---
## 3. Patterns d'erreur par longueur de texte

In [6]:
preds_df['n_words'] = preds_df['text'].str.split().str.len()
preds_df['bin_longueur'] = pd.cut(
    preds_df['n_words'],
    bins=[0, 30, 60, 100, 200, 9999],
    labels=['Très court\n(<30)', 'Court\n(30-60)', 'Moyen\n(60-100)', 'Long\n(100-200)', 'Très long\n(>200)']
)

if pred_cols:
    # Taux d'erreur par longueur
    error_data = []
    for col in pred_cols[:3]:  # 3 premiers modèles
        if preds_df[col].isna().all():
            continue
        tmp = preds_df.copy()
        tmp['erreur'] = tmp[col] != tmp['true_label']
        rate = tmp.groupby('bin_longueur')['erreur'].mean().reset_index()
        rate['model'] = col.replace('pred_', '')
        error_data.append(rate)

    if error_data:
        error_df = pd.concat(error_data)
        fig = px.bar(
            error_df, x='bin_longueur', y='erreur', color='model', barmode='group',
            title='Taux d\'erreur par longueur de mail',
            labels={'erreur': 'Taux d\'erreur', 'bin_longueur': 'Longueur', 'model': 'Modèle'},
            template=TEMPLATE, text_auto='.0%',
        )
        fig.update_yaxes(tickformat='.0%')
        fig.update_layout(height=400)
        fig.show()

---
## 4. Rapport de classification détaillé

Precision, recall et F1 par classe pour chaque modèle.
Permet d'identifier les classes chroniquement mal classifiées.

In [7]:
from sklearn.metrics import precision_recall_fscore_support

def per_class_f1_heatmap(preds_df: pd.DataFrame, pred_cols: list, labels: list) -> go.Figure:
    """Heatmap du F1 par classe et par modèle."""
    f1_matrix = []
    model_names = []

    for col in pred_cols:
        if preds_df[col].isna().all():
            continue
        valid = preds_df.dropna(subset=[col])
        _, _, f1, _ = precision_recall_fscore_support(
            valid['true_label'], valid[col],
            labels=labels, zero_division=0
        )
        f1_matrix.append(f1)
        model_names.append(col.replace('pred_', ''))

    if not f1_matrix:
        return go.Figure()

    fig = go.Figure(go.Heatmap(
        z=f1_matrix,
        x=labels,
        y=model_names,
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=[[f"{v:.0%}" for v in row] for row in f1_matrix],
        texttemplate='%{text}',
        hovertemplate='Modèle: %{y}<br>Classe: %{x}<br>F1: %{z:.0%}<extra></extra>',
    ))
    fig.update_layout(
        title='F1 par classe et par modèle (emails, full data)',
        xaxis_title='Catégorie',
        yaxis_title='Modèle',
        height=350,
        template=TEMPLATE,
    )
    return fig


if pred_cols:
    per_class_f1_heatmap(preds_df, pred_cols, CLASSES).show()
else:
    print("Lance les scripts d'entraînement et sauvegarde les prédictions d'abord.")